# Comparative Analysis of CNN and Transformer Architectures for Semantic Segmentation on SENSATION and CamVid

## Abstract

This project presents a comparative study of three semantic segmentation architectures: FCN-ResNet50, DeepLabV3+, and SegFormer-B0. FCN-ResNet50 and DeepLabV3+ provide two convolutional neural network baselines, while SegFormer-B0 provides a lightweight transformer-based alternative. The experiments are conducted on the SENSATION and CamVid datasets to evaluate segmentation performance in urban scene understanding.

All architectures are trained and evaluated using the same image resolution, preprocessing pipeline, dataset split, optimization settings, and evaluation protocol. The evaluation includes Pixel Accuracy, Mean Intersection over Union, Dice Score, Precision, Recall, and Inference Time. Quantitative results are supported by qualitative visualizations of predicted segmentation masks. The study first compares the two CNN architectures and then examines whether the transformer-based model offers different accuracy, boundary, small-object, or computational characteristics.


## Imports

In [ ]:
from pathlib import Path
import random
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split

from segmentation_data import (
    SegmentationDataset,
    create_training_augmentation,
)

import torchvision
from torchvision.models.segmentation import fcn_resnet50, FCN_ResNet50_Weights

warnings.filterwarnings("ignore")

# Reproducibility
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Device configuration
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Project directories
PROJECT_ROOT = Path(".")
DATA_DIR = PROJECT_ROOT / "data"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
CHECKPOINTS_DIR = PROJECT_ROOT / "checkpoints"
REPORTS_DIR = PROJECT_ROOT / "reports"
CAMVID_CHECKPOINTS_DIR = CHECKPOINTS_DIR / "camvid"
SENSATION_CHECKPOINTS_DIR = CHECKPOINTS_DIR / "sensation"
CAMVID_REPORTS_DIR = REPORTS_DIR / "camvid"
SENSATION_REPORTS_DIR = REPORTS_DIR / "sensation"

for directory in [
    OUTPUTS_DIR,
    CHECKPOINTS_DIR,
    REPORTS_DIR,
    CAMVID_CHECKPOINTS_DIR,
    SENSATION_CHECKPOINTS_DIR,
    CAMVID_REPORTS_DIR,
    SENSATION_REPORTS_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

print(f"PyTorch version: {torch.__version__}")
print(f"Torchvision version: {torchvision.__version__}")
print(f"Using device: {DEVICE}")

if DEVICE.type == "cpu":
    print("Warning: CUDA is not available. Full training may be slow on CPU.")

## Dataset Configuration

The initial experiments are conducted on the CamVid dataset because it is smaller than SENSATION and suitable for validating the complete experimental pipeline. CamVid consists of urban driving scenes with pixel-level semantic annotations. The dataset is already divided into training, validation, and test subsets.

The mask files are stored as class-index PNG images, where each pixel value corresponds to a semantic class. The `Unlabelled` class is treated as an ignored class during loss calculation and metric computation because it does not represent a meaningful semantic category.

In [ ]:
# Dataset 1: CamVid
# Active datasets for the current experiment
ACTIVE_DATASETS = ["sensation"]

# CamVid dataset paths
CAMVID_ROOT = DATA_DIR / "camvid"

CAMVID_IMAGE_DIRS = {
    "train": CAMVID_ROOT / "images" / "train",
    "val": CAMVID_ROOT / "images" / "val",
    "test": CAMVID_ROOT / "images" / "test",
}

CAMVID_MASK_DIRS = {
    "train": CAMVID_ROOT / "masks" / "train",
    "val": CAMVID_ROOT / "masks" / "val",
    "test": CAMVID_ROOT / "masks" / "test",
}

# Standard CamVid 12-class setup used by many semantic segmentation implementations.
# The final class, Unlabelled, is treated as ignore/void.
CAMVID_CLASSES = {
    0: "Sky",
    1: "Building",
    2: "Pole",
    3: "Road",
    4: "Pavement",
    5: "Tree",
    6: "SignSymbol",
    7: "Fence",
    8: "Car",
    9: "Pedestrian",
    10: "Bicyclist",
    11: "Unlabelled",
}

CAMVID_NUM_CLASSES = len(CAMVID_CLASSES)
CAMVID_IGNORE_INDEX = 11

# Training configuration for the small CamVid experiment
IMAGE_SIZE = 384
BATCH_SIZE = 2
NUM_WORKERS = 0
USE_AMP = True
EARLY_STOPPING_PATIENCE = 5
EARLY_STOPPING_MIN_DELTA = 1e-4

print(f"CamVid classes: {CAMVID_NUM_CLASSES}")
print(f"Ignored class index: {CAMVID_IGNORE_INDEX} ({CAMVID_CLASSES[CAMVID_IGNORE_INDEX]})")

# Dataset 2: SENSATION
SENSATION_ROOT = DATA_DIR / "sensation"

SENSATION_IMAGE_DIRS = {
    "train": SENSATION_ROOT / "images" / "train",
    "val": SENSATION_ROOT / "images" / "val",
    "test": SENSATION_ROOT / "images" / "test",
}

SENSATION_MASK_DIRS = {
    "train": SENSATION_ROOT / "masks" / "train",
    "val": SENSATION_ROOT / "masks" / "val",
    "test": SENSATION_ROOT / "masks" / "test",
}

# Official SENSATION mapping from class_colors.csv.
SENSATION_CLASSES = {
    0: "background",
    1: "road",
    2: "sidewalk",
    3: "bikelane",
    4: "person",
    5: "car",
    6: "bicycle",
    7: "traffic sign (front)",
    8: "traffic light",
    9: "obstacle",
}

SENSATION_PALETTE = {
    0: (0, 0, 0),
    1: (0, 0, 255),
    2: (255, 0, 0),
    3: (0, 255, 255),
    4: (255, 183, 0),
    5: (128, 0, 128),
    6: (255, 255, 0),
    7: (255, 128, 128),
    8: (165, 42, 42),
    9: (255, 0, 255),
}

SENSATION_NUM_CLASSES = 10
SENSATION_IGNORE_INDEX = None

# Pixel counts measured from SENSATION training masks only.
SENSATION_CLASS_PIXEL_COUNTS = np.array(
    [
        5767778049,  # background
        724328663,   # road
        2569946899,  # sidewalk
        218227138,   # bikelane
        66036677,    # person
        294476145,   # car
        36265778,    # bicycle
        18708699,    # traffic sign (front)
        9827341,     # traffic light
        301764770,   # obstacle
    ],
    dtype=np.float64,
)

class_frequencies = (
    SENSATION_CLASS_PIXEL_COUNTS
    / SENSATION_CLASS_PIXEL_COUNTS.sum()
)
class_weights = 1.0 / np.log(1.02 + class_frequencies)
class_weights = class_weights / class_weights.mean()

SENSATION_CLASS_WEIGHTS = torch.tensor(
    class_weights,
    dtype=torch.float32,
)

print("SENSATION dataset configuration loaded.")
print(f"SENSATION classes: {SENSATION_NUM_CLASSES}")

In [ ]:
IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg"}
MASK_EXTENSIONS = {".png"}


def list_supported_files(directory: Path, extensions: set[str]) -> list[Path]:
    return sorted(
        path
        for path in directory.iterdir()
        if path.is_file() and path.suffix.lower() in extensions
    )


def index_files_by_stem(
    paths: list[Path],
    file_kind: str,
) -> dict[str, Path]:
    index = {}

    for path in paths:
        if path.stem in index:
            raise ValueError(
                f"Duplicate {file_kind} stem '{path.stem}': "
                f"{index[path.stem].name} and {path.name}"
            )

        index[path.stem] = path

    return index


def verify_segmentation_structure(
    image_dirs: dict[str, Path],
    mask_dirs: dict[str, Path],
    dataset_name: str,
) -> pd.DataFrame:
    rows = []

    for split in ["train", "val", "test"]:
        image_dir = image_dirs[split]
        mask_dir = mask_dirs[split]

        if not image_dir.exists() or not mask_dir.exists():
            rows.append({
                "dataset": dataset_name,
                "split": split,
                "images": 0,
                "masks": 0,
                "matched_pairs": 0,
                "missing_masks": "folder missing",
                "missing_images": "folder missing",
            })
            continue

        image_files = list_supported_files(image_dir, IMAGE_EXTENSIONS)
        mask_files = list_supported_files(mask_dir, MASK_EXTENSIONS)

        image_index = index_files_by_stem(image_files, "image")
        mask_index = index_files_by_stem(mask_files, "mask")

        image_stems = set(image_index)
        mask_stems = set(mask_index)

        missing_masks = sorted(image_stems - mask_stems)
        missing_images = sorted(mask_stems - image_stems)

        rows.append({
            "dataset": dataset_name,
            "split": split,
            "images": len(image_files),
            "masks": len(mask_files),
            "matched_pairs": len(image_stems & mask_stems),
            "missing_masks": len(missing_masks),
            "missing_images": len(missing_images),
        })

    return pd.DataFrame(rows)


def inspect_dataset_mask_values(
    mask_dirs: dict[str, Path],
    dataset_name: str,
) -> list[int] | None:
    existing_mask_files = []

    for split in ["train", "val", "test"]:
        mask_dir = mask_dirs[split]

        if mask_dir.exists():
            existing_mask_files.extend(
                list_supported_files(mask_dir, MASK_EXTENSIONS)
            )

    if not existing_mask_files:
        print(f"{dataset_name}: no PNG mask files found yet.")
        return None

    values = set()

    for mask_path in existing_mask_files:
        mask = cv2.imread(str(mask_path), cv2.IMREAD_UNCHANGED)

        if mask is None:
            raise ValueError(f"Could not read mask: {mask_path}")

        if mask.ndim == 3:
            print(
                f"{dataset_name}: RGB/color mask detected at {mask_path.name}. "
                "A color-to-class mapping will be required."
            )
            return None

        values.update(np.unique(mask).astype(int).tolist())

    return sorted(values)


In [ ]:
# Verify CamVid
camvid_summary = verify_segmentation_structure(
    CAMVID_IMAGE_DIRS,
    CAMVID_MASK_DIRS,
    dataset_name="CamVid",
)

display(camvid_summary)

camvid_mask_values = inspect_dataset_mask_values(
    CAMVID_MASK_DIRS,
    dataset_name="CamVid",
)

print("CamVid mask values:", camvid_mask_values)

# Verify SENSATION
sensation_summary = verify_segmentation_structure(
    SENSATION_IMAGE_DIRS,
    SENSATION_MASK_DIRS,
    dataset_name="SENSATION",
)

display(sensation_summary)

sensation_mask_values = inspect_dataset_mask_values(
    SENSATION_MASK_DIRS,
    dataset_name="SENSATION",
)

print("SENSATION mask values:", sensation_mask_values)

## Visual Verification of Image-Mask Alignment

Before model training, image-mask alignment is verified visually. This step is essential because semantic segmentation uses pixel-level supervision. If an image is paired with the wrong mask, or if the mask is spatially misaligned, the training process becomes invalid.

A sample CamVid image is displayed together with its raw class-index mask, a colorized semantic mask, and an overlay visualization.

In [ ]:
DATASET_CONFIGS = {
    "camvid": {
        "name": "CamVid",
        "image_dirs": CAMVID_IMAGE_DIRS,
        "mask_dirs": CAMVID_MASK_DIRS,
        "classes": CAMVID_CLASSES,
        "num_classes": CAMVID_NUM_CLASSES,
        "ignore_index": CAMVID_IGNORE_INDEX,
        "class_weights": None,
        "checkpoint_dir": CAMVID_CHECKPOINTS_DIR,
        "report_dir": CAMVID_REPORTS_DIR,
        "palette": {
            0: (128, 128, 128),   # Sky
            1: (128, 0, 0),       # Building
            2: (192, 192, 128),   # Pole
            3: (128, 64, 128),    # Road
            4: (60, 40, 222),     # Pavement
            5: (128, 128, 0),     # Tree
            6: (192, 128, 128),   # SignSymbol
            7: (64, 64, 128),     # Fence
            8: (64, 0, 128),      # Car
            9: (64, 64, 0),       # Pedestrian
            10: (0, 128, 192),    # Bicyclist
            11: (0, 0, 0),        # Unlabelled
        },
    },
    "sensation": {
        "name": "SENSATION",
        "image_dirs": SENSATION_IMAGE_DIRS,
        "mask_dirs": SENSATION_MASK_DIRS,
        "classes": SENSATION_CLASSES,
        "num_classes": SENSATION_NUM_CLASSES,
        "ignore_index": SENSATION_IGNORE_INDEX,
        "class_weights": SENSATION_CLASS_WEIGHTS,
        "checkpoint_dir": SENSATION_CHECKPOINTS_DIR,
        "report_dir": SENSATION_REPORTS_DIR,
        "palette": SENSATION_PALETTE,
    },
}

In [ ]:
def load_image_rgb(image_path: Path) -> np.ndarray:
    image_bgr = cv2.imread(str(image_path), cv2.IMREAD_COLOR)

    if image_bgr is None:
        raise ValueError(f"Could not read image: {image_path}")

    return cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)


def load_mask_index(mask_path: Path) -> np.ndarray:
    mask = cv2.imread(str(mask_path), cv2.IMREAD_UNCHANGED)

    if mask is None:
        raise ValueError(f"Could not read mask: {mask_path}")

    if mask.ndim == 3:
        raise ValueError(
            f"Expected class-index mask, but found RGB/color mask: {mask_path}. "
            "A color-to-class conversion mapping is required before training."
        )

    return mask.astype(np.int64)


def colorize_mask(
    mask: np.ndarray,
    palette: dict[int, tuple[int, int, int]],
) -> np.ndarray:
    color_mask = np.zeros((*mask.shape, 3), dtype=np.uint8)

    for class_id, color in palette.items():
        color_mask[mask == class_id] = color

    return color_mask


def overlay_mask_on_image(
    image: np.ndarray,
    color_mask: np.ndarray,
    alpha: float = 0.45,
) -> np.ndarray:
    return cv2.addWeighted(
        image.astype(np.uint8),
        1 - alpha,
        color_mask.astype(np.uint8),
        alpha,
        0,
    )

In [ ]:
def visualize_dataset_sample(
    dataset_key: str,
    split: str = "train",
    sample_index: int = 0,
) -> None:
    config = DATASET_CONFIGS[dataset_key]

    image_dir = config["image_dirs"][split]
    mask_dir = config["mask_dirs"][split]
    palette = config["palette"]

    if not image_dir.exists() or not mask_dir.exists():
        print(f"{config['name']} {split}: image or mask folder does not exist yet.")
        return

    image_paths = list_supported_files(image_dir, IMAGE_EXTENSIONS)
    mask_paths = list_supported_files(mask_dir, MASK_EXTENSIONS)
    mask_index = index_files_by_stem(mask_paths, "mask")

    if not image_paths:
        print(f"{config['name']} {split}: no supported images found.")
        return

    if sample_index >= len(image_paths):
        raise IndexError(
            f"sample_index={sample_index} is out of range for "
            f"{config['name']} {split}, which has {len(image_paths)} images."
        )

    image_path = image_paths[sample_index]
    mask_path = mask_index.get(image_path.stem)

    if mask_path is None:
        raise FileNotFoundError(f"Missing mask for image: {image_path.name}")

    image = load_image_rgb(image_path)
    mask = load_mask_index(mask_path)

    if image.shape[:2] != mask.shape[:2]:
        raise ValueError(
            f"Image and mask dimensions do not match for {image_path.name}: "
            f"image={image.shape[:2]}, mask={mask.shape[:2]}"
        )

    color_mask = colorize_mask(mask, palette)
    overlay = overlay_mask_on_image(image, color_mask)

    print(f"Dataset: {config['name']}")
    print(f"Split: {split}")
    print(f"Image: {image_path.name}")
    print(f"Mask: {mask_path.name}")
    print(f"Image shape: {image.shape}")
    print(f"Mask shape: {mask.shape}")
    print(f"Mask values: {sorted(np.unique(mask).tolist())}")

    fig, axes = plt.subplots(1, 4, figsize=(20, 5))

    axes[0].imshow(image)
    axes[0].set_title("Input Image")
    axes[0].axis("off")

    axes[1].imshow(mask, cmap="tab20")
    axes[1].set_title("Raw Class-Index Mask")
    axes[1].axis("off")

    axes[2].imshow(color_mask)
    axes[2].set_title("Colorized Mask")
    axes[2].axis("off")

    axes[3].imshow(overlay)
    axes[3].set_title("Overlay")
    axes[3].axis("off")

    plt.tight_layout()
    plt.show()


In [ ]:
visualize_dataset_sample("camvid", split="train", sample_index=0)

In [ ]:
visualize_dataset_sample("sensation", split="train", sample_index=0)

## PyTorch Dataset and DataLoader

A custom PyTorch `Dataset` is used to load semantic segmentation image-mask pairs. Each image is resized, normalized, and converted into a tensor. Each mask is resized using nearest-neighbor interpolation to preserve integer class labels.

The same dataset class is reused for CamVid and SENSATION. FCN-ResNet50, DeepLabV3+, and SegFormer-B0 therefore receive identical input tensors and masks, which supports a controlled comparison between the CNN and transformer-based architectures.


Dataset loading, synchronized augmentation, ImageNet normalization, and optional RAM caching are implemented in `segmentation_data.py`. Keeping this reusable pipeline outside the notebook reduces duplication and ensures that all architectures receive identical inputs.

In [ ]:
def create_dataloaders(
    dataset_key: str,
    image_size: int = IMAGE_SIZE,
    batch_size: int = BATCH_SIZE,
    num_workers: int = NUM_WORKERS,
) -> dict[str, DataLoader]:
    config = DATASET_CONFIGS[dataset_key]

    dataloaders = {}

    for split in ["train", "val", "test"]:
        augmentation = (
            create_training_augmentation()
            if split == "train"
            else None
        )

        dataset = SegmentationDataset(
            image_dir=config["image_dirs"][split],
            mask_dir=config["mask_dirs"][split],
            image_size=image_size,
            num_classes=config["num_classes"],
            ignore_index=config["ignore_index"],
            augmentation=augmentation,
            cache_in_memory=(
                dataset_key == "sensation" and split == "train"
            ),
        )

        dataloaders[split] = DataLoader(
            dataset,
            batch_size=batch_size,
            shuffle=(split == "train"),
            num_workers=num_workers,
            pin_memory=(DEVICE.type == "cuda"),
            drop_last=(split == "train"),
        )

        print(
            f"{config['name']} {split}: "
            f"{len(dataset)} samples, "
            f"{len(dataloaders[split])} batches"
        )

    return dataloaders

In [ ]:
camvid_loaders = create_dataloaders("camvid")
sensation_loaders = create_dataloaders("sensation")

In [ ]:
images, masks = next(iter(camvid_loaders["train"]))

print("Image batch shape:", images.shape)
print("Mask batch shape:", masks.shape)
print("Image dtype:", images.dtype)
print("Mask dtype:", masks.dtype)
print("Mask min:", masks.min().item())
print("Mask max:", masks.max().item())

In [ ]:
images, masks = next(iter(sensation_loaders["train"]))

print("Image batch shape:", images.shape)
print("Mask batch shape:", masks.shape)
print("Image dtype:", images.dtype)
print("Mask dtype:", masks.dtype)
print("Mask min:", masks.min().item())
print("Mask max:", masks.max().item())

## Model Architectures

Three semantic segmentation architectures are compared under the same experimental setting.

**FCN-ResNet50** is a convolutional architecture that uses a ResNet50 backbone and replaces fully connected classification layers with convolutional layers for dense pixel-level prediction. Its comparatively simple segmentation head provides a useful CNN baseline, although repeated downsampling and upsampling can reduce boundary precision.

**DeepLabV3+** is also convolutional and uses a ResNet50 encoder. Atrous Spatial Pyramid Pooling captures context at multiple receptive-field scales, while the decoder refines the segmentation output. These components are intended to improve multi-scale scene understanding and object boundaries.

**SegFormer-B0** is a lightweight transformer-based segmentation model. Its MiT-B0 hierarchical encoder combines local and global information at multiple feature scales, and its compact decoder fuses these representations without positional encodings. It provides a contrasting architectural approach while remaining computationally practical for the available hardware.

ImageNet-pretrained encoder weights are used for all three models. The output layer of each architecture is configured for the dataset-specific number of semantic classes.


In [ ]:
import segmentation_models_pytorch as smp


def create_fcn_resnet50(
    num_classes: int,
    pretrained: bool = True,
) -> nn.Module:
    if pretrained:
        weights = FCN_ResNet50_Weights.DEFAULT
    else:
        weights = None

    model = fcn_resnet50(weights=weights)

    # Replace the final classifier layer so the model predicts the correct number of classes.
    model.classifier[4] = nn.Conv2d(
        in_channels=512,
        out_channels=num_classes,
        kernel_size=1,
    )

    # Replace auxiliary classifier if it exists.
    if model.aux_classifier is not None:
        model.aux_classifier[4] = nn.Conv2d(
            in_channels=256,
            out_channels=num_classes,
            kernel_size=1,
        )

    return model


def create_deeplabv3plus(
    num_classes: int,
    encoder_name: str = "resnet50",
    encoder_weights: str | None = "imagenet",
) -> nn.Module:
    return smp.DeepLabV3Plus(
        encoder_name=encoder_name,
        encoder_weights=encoder_weights,
        in_channels=3,
        classes=num_classes,
    )


def create_segformer_b0(
    num_classes: int,
    encoder_weights: str | None = "imagenet",
) -> nn.Module:
    return smp.Segformer(
        encoder_name="mit_b0",
        encoder_weights=encoder_weights,
        in_channels=3,
        classes=num_classes,
    )


In [ ]:
def count_trainable_parameters(model: nn.Module) -> int:
    return sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)


def build_models_for_dataset(dataset_key: str) -> dict[str, nn.Module]:
    config = DATASET_CONFIGS[dataset_key]
    num_classes = config["num_classes"]

    models = {
        "FCN-ResNet50": create_fcn_resnet50(num_classes=num_classes),
        "DeepLabV3+": create_deeplabv3plus(num_classes=num_classes),
        "SegFormer-B0": create_segformer_b0(num_classes=num_classes),
    }

    for model_name, model in models.items():
        models[model_name] = model.to(DEVICE)
        parameters = count_trainable_parameters(models[model_name])
        print(f"{model_name}: {parameters:,} trainable parameters")

    return models

In [ ]:
camvid_models = build_models_for_dataset("camvid")
sensation_models = build_models_for_dataset("sensation")

## Loss Function, Optimizer, and Evaluation Metrics

All architectures use the same optimizer and loss configuration within each dataset. CamVid retains standard Cross Entropy Loss. SENSATION uses a 50:50 combination of logarithmically weighted Cross Entropy and multiclass Dice loss to reduce domination by frequent classes while improving overlap for rare classes.

SENSATION class weights are derived only from training-mask pixel frequencies. Validation and test masks do not influence the loss weights. Aggregate and per-class evaluation metrics remain unweighted and are calculated from a dataset-level confusion matrix.


In [ ]:
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4


class WeightedCrossEntropyDiceLoss(nn.Module):
    def __init__(
        self,
        class_weights: torch.Tensor,
        ignore_index: int | None = None,
        cross_entropy_weight: float = 0.5,
        dice_weight: float = 0.5,
    ):
        super().__init__()
        self.cross_entropy_weight = cross_entropy_weight
        self.dice_weight = dice_weight

        cross_entropy_ignore_index = (
            ignore_index if ignore_index is not None else -100
        )
        self.cross_entropy = nn.CrossEntropyLoss(
            weight=class_weights,
            ignore_index=cross_entropy_ignore_index,
        )
        self.dice = smp.losses.DiceLoss(
            mode="multiclass",
            from_logits=True,
            ignore_index=ignore_index,
        )

    def forward(
        self,
        predictions: torch.Tensor,
        targets: torch.Tensor,
    ) -> torch.Tensor:
        cross_entropy_loss = self.cross_entropy(predictions, targets)
        dice_loss = self.dice(predictions, targets)

        return (
            self.cross_entropy_weight * cross_entropy_loss
            + self.dice_weight * dice_loss
        )


def create_loss_function(
    ignore_index: int | None = None,
    class_weights: torch.Tensor | None = None,
) -> nn.Module:
    if class_weights is None:
        if ignore_index is None:
            return nn.CrossEntropyLoss()

        return nn.CrossEntropyLoss(ignore_index=ignore_index)

    return WeightedCrossEntropyDiceLoss(
        class_weights=class_weights,
        ignore_index=ignore_index,
    )


def create_optimizer(model: nn.Module) -> torch.optim.Optimizer:
    return torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )


def create_scheduler(optimizer: torch.optim.Optimizer):
    return torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=3,
    )


In [ ]:
def update_confusion_matrix(
    confusion_matrix: torch.Tensor,
    predictions: torch.Tensor,
    targets: torch.Tensor,
    num_classes: int,
    ignore_index: int | None = None,
) -> torch.Tensor:
    predictions = predictions.reshape(-1).long()
    targets = targets.reshape(-1).long()

    valid_mask = (
        (targets >= 0)
        & (targets < num_classes)
        & (predictions >= 0)
        & (predictions < num_classes)
    )

    if ignore_index is not None:
        valid_mask &= targets != ignore_index

    if not torch.any(valid_mask):
        return confusion_matrix

    encoded = (
        targets[valid_mask] * num_classes
        + predictions[valid_mask]
    )
    batch_matrix = torch.bincount(
        encoded,
        minlength=num_classes * num_classes,
    ).reshape(num_classes, num_classes)

    confusion_matrix += batch_matrix.to(confusion_matrix.device)
    return confusion_matrix


def compute_metrics_from_confusion_matrix(
    confusion_matrix: torch.Tensor,
    ignore_index: int | None = None,
) -> dict[str, float | list[float]]:
    matrix = confusion_matrix.detach().cpu().to(torch.float64)
    num_classes = matrix.shape[0]

    true_positive = torch.diag(matrix)
    target_support = matrix.sum(dim=1)
    prediction_support = matrix.sum(dim=0)

    false_positive = prediction_support - true_positive
    false_negative = target_support - true_positive
    union = true_positive + false_positive + false_negative

    valid_classes = target_support > 0
    if ignore_index is not None and 0 <= ignore_index < num_classes:
        valid_classes[ignore_index] = False

    per_class_iou = torch.full((num_classes,), torch.nan, dtype=torch.float64)
    per_class_dice = torch.full((num_classes,), torch.nan, dtype=torch.float64)
    per_class_precision = torch.zeros(num_classes, dtype=torch.float64)
    per_class_recall = torch.zeros(num_classes, dtype=torch.float64)

    per_class_iou[valid_classes] = (
        true_positive[valid_classes] / union[valid_classes]
    )
    per_class_dice[valid_classes] = (
        2 * true_positive[valid_classes]
        / (prediction_support[valid_classes] + target_support[valid_classes])
    )

    predicted_classes = prediction_support > 0
    per_class_precision[predicted_classes] = (
        true_positive[predicted_classes]
        / prediction_support[predicted_classes]
    )
    per_class_recall[valid_classes] = (
        true_positive[valid_classes] / target_support[valid_classes]
    )

    total_valid_pixels = matrix.sum()
    pixel_accuracy = (
        true_positive.sum() / total_valid_pixels
        if total_valid_pixels > 0
        else torch.tensor(0.0, dtype=torch.float64)
    )

    if torch.any(valid_classes):
        mean_iou = per_class_iou[valid_classes].mean()
        dice_score = per_class_dice[valid_classes].mean()
        precision = per_class_precision[valid_classes].mean()
        recall = per_class_recall[valid_classes].mean()
    else:
        mean_iou = torch.tensor(0.0, dtype=torch.float64)
        dice_score = torch.tensor(0.0, dtype=torch.float64)
        precision = torch.tensor(0.0, dtype=torch.float64)
        recall = torch.tensor(0.0, dtype=torch.float64)

    return {
        "pixel_accuracy": float(pixel_accuracy),
        "mean_iou": float(mean_iou),
        "dice_score": float(dice_score),
        "precision": float(precision),
        "recall": float(recall),
        "per_class_iou": per_class_iou.tolist(),
    }


def compute_segmentation_metrics(
    predictions: torch.Tensor,
    targets: torch.Tensor,
    num_classes: int,
    ignore_index: int | None = None,
) -> dict[str, float | list[float]]:
    confusion_matrix = torch.zeros(
        (num_classes, num_classes),
        dtype=torch.int64,
        device=predictions.device,
    )
    update_confusion_matrix(
        confusion_matrix=confusion_matrix,
        predictions=predictions,
        targets=targets,
        num_classes=num_classes,
        ignore_index=ignore_index,
    )
    return compute_metrics_from_confusion_matrix(
        confusion_matrix=confusion_matrix,
        ignore_index=ignore_index,
    )


In [ ]:
images, masks = next(iter(camvid_loaders["train"]))

images = images.to(DEVICE)
masks = masks.to(DEVICE)

model = camvid_models["FCN-ResNet50"]
model.eval()

with torch.no_grad():
    outputs = model(images)

    if isinstance(outputs, dict):
        outputs = outputs["out"]

    predictions = torch.argmax(outputs, dim=1)

metrics = compute_segmentation_metrics(
    predictions=predictions,
    targets=masks,
    num_classes=CAMVID_NUM_CLASSES,
    ignore_index=CAMVID_IGNORE_INDEX,
)

metrics

In [ ]:
images, masks = next(iter(sensation_loaders["train"]))

images = images.to(DEVICE)
masks = masks.to(DEVICE)

model = sensation_models["FCN-ResNet50"]
model.eval()

with torch.no_grad():
    outputs = model(images)

    if isinstance(outputs, dict):
        outputs = outputs["out"]

    predictions = torch.argmax(outputs, dim=1)

metrics = compute_segmentation_metrics(
    predictions=predictions,
    targets=masks,
    num_classes=SENSATION_NUM_CLASSES,
    ignore_index=SENSATION_IGNORE_INDEX,
)

metrics

## Training and Validation Procedure

The training loop updates model parameters using the training set, while the validation loop evaluates generalization performance without updating parameters. For each epoch, training loss, validation loss, and validation metrics are recorded.

The best model checkpoint is selected based on validation Mean IoU because mIoU is a standard metric for semantic segmentation and is less biased by dominant classes than pixel accuracy.

In [ ]:
def get_model_output(model: nn.Module, images: torch.Tensor) -> torch.Tensor:
    outputs = model(images)

    # Torchvision segmentation models return a dictionary.
    # segmentation_models_pytorch models return the tensor directly.
    if isinstance(outputs, dict):
        outputs = outputs["out"]

    return outputs

In [ ]:
def train_one_epoch(
    model: nn.Module,
    dataloader: DataLoader,
    loss_function: nn.Module,
    optimizer: torch.optim.Optimizer,
    scaler: torch.amp.GradScaler,
    device: torch.device,
) -> float:
    model.train()
    running_loss = 0.0
    amp_enabled = USE_AMP and device.type == "cuda"

    for images, masks in dataloader:
        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(
            device_type=device.type,
            enabled=amp_enabled,
        ):
            outputs = get_model_output(model, images)
            loss = loss_function(outputs, masks)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()

    return running_loss / len(dataloader)


In [ ]:
def evaluate_model(
    model: nn.Module,
    dataloader: DataLoader,
    loss_function: nn.Module,
    device: torch.device,
    num_classes: int,
    ignore_index: int | None = None,
) -> dict[str, float | list[float]]:
    model.eval()
    amp_enabled = USE_AMP and device.type == "cuda"

    running_loss = 0.0
    total_samples = 0
    confusion_matrix = torch.zeros(
        (num_classes, num_classes),
        dtype=torch.int64,
        device=device,
    )

    with torch.no_grad():
        for images, masks in dataloader:
            images = images.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)

            with torch.amp.autocast(
                device_type=device.type,
                enabled=amp_enabled,
            ):
                outputs = get_model_output(model, images)
                loss = loss_function(outputs, masks)

            predictions = torch.argmax(outputs, dim=1)

            batch_size = images.size(0)
            running_loss += loss.item() * batch_size
            total_samples += batch_size

            update_confusion_matrix(
                confusion_matrix=confusion_matrix,
                predictions=predictions,
                targets=masks,
                num_classes=num_classes,
                ignore_index=ignore_index,
            )

    results = compute_metrics_from_confusion_matrix(
        confusion_matrix=confusion_matrix,
        ignore_index=ignore_index,
    )
    results["loss"] = running_loss / total_samples

    return results


In [ ]:
def train_model(
    model: nn.Module,
    model_name: str,
    dataset_key: str,
    dataloaders: dict[str, DataLoader],
    num_epochs: int,
) -> pd.DataFrame:
    config = DATASET_CONFIGS[dataset_key]

    loss_function = create_loss_function(
        ignore_index=config["ignore_index"],
        class_weights=config.get("class_weights"),
    ).to(DEVICE)
    optimizer = create_optimizer(model)
    scheduler = create_scheduler(optimizer)
    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=USE_AMP and DEVICE.type == "cuda",
    )

    best_val_miou = -1.0
    epochs_without_improvement = 0
    history = []

    checkpoint_dir = config.get("checkpoint_dir", CHECKPOINTS_DIR)
    report_dir = config.get("report_dir", REPORTS_DIR)
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    report_dir.mkdir(parents=True, exist_ok=True)

    artifact_name = model_name.replace(" ", "_").replace("+", "plus")
    checkpoint_path = checkpoint_dir / f"{dataset_key}_{artifact_name}_best.pth"

    for epoch in range(num_epochs):
        start_time = time.time()

        train_loss = train_one_epoch(
            model=model,
            dataloader=dataloaders["train"],
            loss_function=loss_function,
            optimizer=optimizer,
            scaler=scaler,
            device=DEVICE,
        )

        val_results = evaluate_model(
            model=model,
            dataloader=dataloaders["val"],
            loss_function=loss_function,
            device=DEVICE,
            num_classes=config["num_classes"],
            ignore_index=config["ignore_index"],
        )

        scheduler.step(val_results["mean_iou"])
        epoch_time = time.time() - start_time

        row = {
            "dataset": config["name"],
            "model": model_name,
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "val_loss": val_results["loss"],
            "val_pixel_accuracy": val_results["pixel_accuracy"],
            "val_mean_iou": val_results["mean_iou"],
            "val_dice_score": val_results["dice_score"],
            "val_precision": val_results["precision"],
            "val_recall": val_results["recall"],
            "epoch_time_sec": epoch_time,
        }
        history.append(row)

        print(
            f"[{config['name']} | {model_name}] "
            f"Epoch {epoch + 1}/{num_epochs} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val Loss: {val_results['loss']:.4f} | "
            f"Val mIoU: {val_results['mean_iou']:.4f} | "
            f"Time: {epoch_time:.1f}s"
        )

        improved = (
            val_results["mean_iou"]
            > best_val_miou + EARLY_STOPPING_MIN_DELTA
        )

        if improved:
            best_val_miou = val_results["mean_iou"]
            epochs_without_improvement = 0

            torch.save(
                {
                    "model_state_dict": model.state_dict(),
                    "model_name": model_name,
                    "dataset_key": dataset_key,
                    "epoch": epoch + 1,
                    "best_val_miou": best_val_miou,
                    "class_names": config["classes"],
                    "use_amp": USE_AMP,
                },
                checkpoint_path,
            )
            print(f"Saved best checkpoint: {checkpoint_path}")
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
            print(
                "Early stopping: validation mIoU did not improve for "
                f"{EARLY_STOPPING_PATIENCE} epochs."
            )
            break

    history_df = pd.DataFrame(history)
    history_csv_path = report_dir / f"{dataset_key}_{artifact_name}_history.csv"
    history_df.to_csv(history_csv_path, index=False)

    return history_df


In [ ]:
NUM_EPOCHS = 25

camvid_fcn_history = train_model(
    model=camvid_models["FCN-ResNet50"],
    model_name="FCN-ResNet50",
    dataset_key="camvid",
    dataloaders=camvid_loaders,
    num_epochs=NUM_EPOCHS,
)

display(camvid_fcn_history)

In [ ]:
NUM_EPOCHS = 25
camvid_deeplab_history = train_model(
    model=camvid_models["DeepLabV3+"],
    model_name="DeepLabV3+",
    dataset_key="camvid",
    dataloaders=camvid_loaders,
    num_epochs=NUM_EPOCHS,
)

display(camvid_deeplab_history)

In [ ]:
camvid_segformer_history = train_model(
    model=camvid_models["SegFormer-B0"],
    model_name="SegFormer-B0",
    dataset_key="camvid",
    dataloaders=camvid_loaders,
    num_epochs=NUM_EPOCHS,
)

display(camvid_segformer_history)


## Training and Validation Curves

Training and validation curves are used to analyze optimization behavior. The loss curves indicate whether the models are learning from the training data, while validation metrics show how well each model generalizes to unseen validation images.

These plots are used for academic comparison and are saved to the reports directory.

In [ ]:
camvid_training_history = pd.concat(
    [
        camvid_fcn_history,
        camvid_deeplab_history,
        camvid_segformer_history,
    ],
    ignore_index=True,
)

display(camvid_training_history)


In [ ]:
def plot_training_curves(
    history_df: pd.DataFrame,
    dataset_key: str,
) -> None:
    dataset_name = DATASET_CONFIGS[dataset_key]["name"]

    metrics_to_plot = [
        ("train_loss", "Training Loss"),
        ("val_loss", "Validation Loss"),
        ("val_mean_iou", "Validation Mean IoU"),
        ("val_pixel_accuracy", "Validation Pixel Accuracy"),
        ("val_dice_score", "Validation Dice Score"),
    ]

    fig, axes = plt.subplots(1, len(metrics_to_plot), figsize=(24, 4))

    for axis, (metric_column, metric_title) in zip(axes, metrics_to_plot):
        for model_name in history_df["model"].unique():
            model_history = history_df[history_df["model"] == model_name]

            axis.plot(
                model_history["epoch"],
                model_history[metric_column],
                marker="o",
                label=model_name,
            )

        axis.set_title(metric_title)
        axis.set_xlabel("Epoch")
        axis.grid(True, alpha=0.3)

        if metric_column not in ["train_loss", "val_loss"]:
            axis.set_ylim(0, 1)

    axes[-1].legend(loc="lower right")

    plt.suptitle(f"{dataset_name}: Training and Validation Curves", fontsize=14)
    plt.tight_layout()

    report_dir = DATASET_CONFIGS[dataset_key].get("report_dir", REPORTS_DIR)
    report_dir.mkdir(parents=True, exist_ok=True)
    output_path = report_dir / f"{dataset_key}_training_validation_curves.png"
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()

    print(f"Saved curve figure to: {output_path}")

In [ ]:
plot_training_curves(
    history_df=camvid_training_history,
    dataset_key="camvid",
)

In [ ]:
def summarize_best_validation_results(history_df: pd.DataFrame) -> pd.DataFrame:
    best_rows = []

    for model_name in history_df["model"].unique():
        model_history = history_df[history_df["model"] == model_name]
        best_row = model_history.loc[model_history["val_mean_iou"].idxmax()]
        best_rows.append(best_row)

    summary = pd.DataFrame(best_rows)

    selected_columns = [
        "dataset",
        "model",
        "epoch",
        "val_loss",
        "val_pixel_accuracy",
        "val_mean_iou",
        "val_dice_score",
        "val_precision",
        "val_recall",
        "epoch_time_sec",
    ]

    summary = summary[selected_columns].sort_values(
        by="val_mean_iou",
        ascending=False,
    )

    return summary.reset_index(drop=True)

In [ ]:
camvid_best_validation_summary = summarize_best_validation_results(camvid_training_history)

display(camvid_best_validation_summary)

summary_path = CAMVID_REPORTS_DIR / "camvid_best_validation_summary.csv"
camvid_best_validation_summary.to_csv(summary_path, index=False)

print(f"Saved validation summary to: {summary_path}")

## Test-Set Evaluation

After model training, the best checkpoint for each architecture is loaded and evaluated on the test set. The test set is not used during optimization or checkpoint selection.

Metrics are calculated from one confusion matrix accumulated over the complete dataset split rather than by averaging independent batch scores. This produces dataset-level Pixel Accuracy, Mean IoU, Dice Score, Precision, Recall, and per-class IoU. Inference time is measured under the same hardware and input settings for all architectures.


In [ ]:
def load_best_checkpoint(
    model: nn.Module,
    checkpoint_path: Path,
    device: torch.device,
) -> nn.Module:
    if not checkpoint_path.exists():
        raise FileNotFoundError(f"Checkpoint not found: {checkpoint_path}")

    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.to(device)
    model.eval()

    print(
        f"Loaded checkpoint: {checkpoint_path} "
        f"(epoch={checkpoint['epoch']}, best_val_mIoU={checkpoint['best_val_miou']:.4f})"
    )

    return model

In [ ]:
def measure_batch_inference_time(
    model: nn.Module,
    images: torch.Tensor,
    device: torch.device,
) -> float:
    model.eval()

    images = images.to(device, non_blocking=True)
    amp_enabled = USE_AMP and device.type == "cuda"

    if device.type == "cuda":
        torch.cuda.synchronize()

    start_time = time.time()

    with torch.no_grad():
        with torch.amp.autocast(
            device_type=device.type,
            enabled=amp_enabled,
        ):
            _ = get_model_output(model, images)

    if device.type == "cuda":
        torch.cuda.synchronize()

    elapsed_time = time.time() - start_time

    return elapsed_time / images.size(0)

In [ ]:
def evaluate_on_test_set(
    model: nn.Module,
    model_name: str,
    dataset_key: str,
    dataloader: DataLoader,
) -> dict[str, object]:
    config = DATASET_CONFIGS[dataset_key]

    loss_function = create_loss_function(
        ignore_index=config["ignore_index"],
        class_weights=config.get("class_weights"),
    ).to(DEVICE)

    test_results = evaluate_model(
        model=model,
        dataloader=dataloader,
        loss_function=loss_function,
        device=DEVICE,
        num_classes=config["num_classes"],
        ignore_index=config["ignore_index"],
    )

    inference_times = []

    with torch.no_grad():
        for images, _ in dataloader:
            batch_time = measure_batch_inference_time(
                model=model,
                images=images,
                device=DEVICE,
            )
            inference_times.append(batch_time)

    return {
        "dataset": config["name"],
        "model": model_name,
        "test_loss": test_results["loss"],
        "test_pixel_accuracy": test_results["pixel_accuracy"],
        "test_mean_iou": test_results["mean_iou"],
        "test_dice_score": test_results["dice_score"],
        "test_precision": test_results["precision"],
        "test_recall": test_results["recall"],
        "inference_time_sec_per_image": float(np.mean(inference_times)),
        "per_class_iou": test_results["per_class_iou"],
    }


In [ ]:
camvid_best_fcn_path = CAMVID_CHECKPOINTS_DIR / "camvid_FCN-ResNet50_best.pth"
camvid_best_deeplab_path = CAMVID_CHECKPOINTS_DIR / "camvid_DeepLabV3plus_best.pth"
camvid_best_segformer_path = CAMVID_CHECKPOINTS_DIR / "camvid_SegFormer-B0_best.pth"

camvid_models["FCN-ResNet50"] = load_best_checkpoint(
    model=camvid_models["FCN-ResNet50"],
    checkpoint_path=camvid_best_fcn_path,
    device=DEVICE,
)

camvid_models["DeepLabV3+"] = load_best_checkpoint(
    model=camvid_models["DeepLabV3+"],
    checkpoint_path=camvid_best_deeplab_path,
    device=DEVICE,
)

camvid_models["SegFormer-B0"] = load_best_checkpoint(
    model=camvid_models["SegFormer-B0"],
    checkpoint_path=camvid_best_segformer_path,
    device=DEVICE,
)


In [ ]:
camvid_test_results = []
camvid_per_class_iou_rows = []
camvid_config = DATASET_CONFIGS["camvid"]

for model_name, model in camvid_models.items():
    result = evaluate_on_test_set(
        model=model,
        model_name=model_name,
        dataset_key="camvid",
        dataloader=camvid_loaders["test"],
    )

    per_class_iou = result.pop("per_class_iou")
    camvid_test_results.append(result)

    for class_id, class_iou in enumerate(per_class_iou):
        if class_id == camvid_config["ignore_index"]:
            continue

        camvid_per_class_iou_rows.append({
            "dataset": camvid_config["name"],
            "model": model_name,
            "class_id": class_id,
            "class_name": camvid_config["classes"][class_id],
            "iou": class_iou,
        })

camvid_test_results_df = pd.DataFrame(camvid_test_results)
camvid_test_results_df = camvid_test_results_df.sort_values(
    by="test_mean_iou",
    ascending=False,
).reset_index(drop=True)

camvid_per_class_iou_df = pd.DataFrame(camvid_per_class_iou_rows)

display(camvid_test_results_df)
display(camvid_per_class_iou_df)

test_results_path = CAMVID_REPORTS_DIR / "camvid_test_results.csv"
per_class_path = CAMVID_REPORTS_DIR / "camvid_per_class_iou.csv"
camvid_test_results_df.to_csv(test_results_path, index=False)
camvid_per_class_iou_df.to_csv(per_class_path, index=False)

print(f"Saved test results to: {test_results_path}")
print(f"Saved per-class IoU to: {per_class_path}")


## Qualitative Segmentation Results

Qualitative results are visualized to compare the spatial behavior of FCN-ResNet50, DeepLabV3+, and SegFormer-B0. While quantitative metrics summarize performance numerically, prediction masks reveal model behavior around object boundaries, thin structures, small objects, and visually ambiguous regions.

For each selected test image, the input image, ground-truth mask, and predictions from all three architectures are displayed side by side.


In [ ]:
def predict_mask(
    model: nn.Module,
    image_tensor: torch.Tensor,
    device: torch.device,
) -> np.ndarray:
    model.eval()

    with torch.no_grad():
        image_tensor = image_tensor.unsqueeze(0).to(device)
        output = get_model_output(model, image_tensor)
        prediction = torch.argmax(output, dim=1)

    return prediction.squeeze(0).cpu().numpy().astype(np.int64)

In [ ]:
def denormalize_image(image_tensor: torch.Tensor) -> np.ndarray:
    image = image_tensor.detach().cpu().permute(1, 2, 0).numpy()

    mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    std = np.array([0.229, 0.224, 0.225], dtype=np.float32)

    image = (image * std) + mean
    image = np.clip(image, 0, 1)

    return (image * 255).astype(np.uint8)

In [ ]:
def visualize_model_predictions(
    dataset_key: str,
    dataloader: DataLoader,
    models: dict[str, nn.Module],
    sample_count: int = 4,
) -> None:
    config = DATASET_CONFIGS[dataset_key]
    palette = config["palette"]

    dataset = dataloader.dataset

    fig, axes = plt.subplots(
        sample_count,
        2 + len(models),
        figsize=(5 * (2 + len(models)), 4 * sample_count),
    )

    if sample_count == 1:
        axes = np.expand_dims(axes, axis=0)

    for row_index in range(sample_count):
        image_tensor, mask_tensor = dataset[row_index]

        image = denormalize_image(image_tensor)
        ground_truth = mask_tensor.numpy()
        ground_truth_color = colorize_mask(ground_truth, palette)

        axes[row_index, 0].imshow(image)
        axes[row_index, 0].set_title("Input Image")
        axes[row_index, 0].axis("off")

        axes[row_index, 1].imshow(ground_truth_color)
        axes[row_index, 1].set_title("Ground Truth")
        axes[row_index, 1].axis("off")

        for model_col, (model_name, model) in enumerate(models.items(), start=2):
            prediction = predict_mask(
                model=model,
                image_tensor=image_tensor,
                device=DEVICE,
            )

            prediction_color = colorize_mask(prediction, palette)

            axes[row_index, model_col].imshow(prediction_color)
            axes[row_index, model_col].set_title(model_name)
            axes[row_index, model_col].axis("off")

    plt.tight_layout()

    report_dir = config.get("report_dir", REPORTS_DIR)
    report_dir.mkdir(parents=True, exist_ok=True)
    output_path = report_dir / f"{dataset_key}_qualitative_predictions.png"
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()

    print(f"Saved qualitative prediction figure to: {output_path}")

In [ ]:
visualize_model_predictions(
    dataset_key="camvid",
    dataloader=camvid_loaders["test"],
    models=camvid_models,
    sample_count=4,
)

## Quantitative Metric Comparison

The final quantitative comparison is summarized using bar charts for the main test-set metrics. These visualizations first support comparison of the two CNN architectures and then show how their performance and inference cost differ from the transformer-based SegFormer-B0 model.


In [ ]:
def plot_test_metric_comparison(
    results_df: pd.DataFrame,
    dataset_key: str,
) -> None:
    dataset_name = DATASET_CONFIGS[dataset_key]["name"]

    metric_columns = [
        ("test_pixel_accuracy", "Pixel Accuracy"),
        ("test_mean_iou", "Mean IoU"),
        ("test_dice_score", "Dice Score"),
        ("test_precision", "Precision"),
        ("test_recall", "Recall"),
        ("inference_time_sec_per_image", "Inference Time / Image (s)"),
    ]

    fig, axes = plt.subplots(2, 3, figsize=(16, 9))
    bar_colors = ["#4C78A8", "#F58518", "#54A24B"]
    axes = axes.flatten()

    for axis, (column, title) in zip(axes, metric_columns):
        axis.bar(
            results_df["model"],
            results_df[column],
            color=bar_colors[:len(results_df)],
        )

        axis.set_title(title)
        axis.set_xlabel("Model")
        axis.grid(axis="y", alpha=0.3)

        if column != "inference_time_sec_per_image":
            axis.set_ylim(0, 1)

        for index, value in enumerate(results_df[column]):
            axis.text(
                index,
                value,
                f"{value:.4f}",
                ha="center",
                va="bottom",
                fontsize=9,
            )

    plt.suptitle(f"{dataset_name}: Test Metric Comparison", fontsize=14)
    plt.tight_layout()

    report_dir = DATASET_CONFIGS[dataset_key].get("report_dir", REPORTS_DIR)
    report_dir.mkdir(parents=True, exist_ok=True)
    output_path = report_dir / f"{dataset_key}_test_metric_comparison.png"
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()

    print(f"Saved metric comparison chart to: {output_path}")

In [ ]:
plot_test_metric_comparison(
    results_df=camvid_test_results_df,
    dataset_key="camvid",
)

In [ ]:
display(
    camvid_test_results_df[
        [
            "model",
            "test_pixel_accuracy",
            "test_mean_iou",
            "test_dice_score",
            "test_precision",
            "test_recall",
            "inference_time_sec_per_image",
        ]
    ]
)

## SENSATION Semantic Segmentation

All three architectures are retrained using the official 10-class mapping, synchronized augmentation, logarithmically weighted Cross Entropy plus Dice loss, CUDA automatic mixed precision, global validation metrics, and early stopping. These settings are identical across the three SENSATION models.

SENSATION checkpoints and reports are stored in dataset-specific folders under `checkpoints/sensation` and `reports/sensation`.

### SENSATION Training


In [ ]:
NUM_EPOCHS = 25

sensation_fcn_history = train_model(
    model=sensation_models["FCN-ResNet50"],
    model_name="FCN-ResNet50",
    dataset_key="sensation",
    dataloaders=sensation_loaders,
    num_epochs=NUM_EPOCHS,
)

display(sensation_fcn_history)

In [ ]:
sensation_deeplab_history = train_model(
    model=sensation_models["DeepLabV3+"],
    model_name="DeepLabV3+",
    dataset_key="sensation",
    dataloaders=sensation_loaders,
    num_epochs=NUM_EPOCHS,
)

display(sensation_deeplab_history)


In [ ]:
sensation_segformer_history = train_model(
    model=sensation_models["SegFormer-B0"],
    model_name="SegFormer-B0",
    dataset_key="sensation",
    dataloaders=sensation_loaders,
    num_epochs=NUM_EPOCHS,
)

display(sensation_segformer_history)


### SENSATION Training and Validation Analysis

The three training histories are combined only after all SENSATION models have completed training. The best validation row for each architecture is selected by validation Mean IoU.


In [ ]:
sensation_training_history = pd.concat(
    [
        sensation_fcn_history,
        sensation_deeplab_history,
        sensation_segformer_history,
    ],
    ignore_index=True,
)

display(sensation_training_history)


In [ ]:
plot_training_curves(
    history_df=sensation_training_history,
    dataset_key="sensation",
)


In [ ]:
sensation_best_validation_summary = summarize_best_validation_results(
    sensation_training_history
)

display(sensation_best_validation_summary)

sensation_summary_path = SENSATION_REPORTS_DIR / "sensation_best_validation_summary.csv"
sensation_best_validation_summary.to_csv(sensation_summary_path, index=False)

print(f"Saved validation summary to: {sensation_summary_path}")


### SENSATION Test-Set Evaluation

The best validation checkpoint for each architecture is loaded before test evaluation. The SENSATION test set remains excluded from training, optimization, and checkpoint selection.


In [ ]:
sensation_best_fcn_path = SENSATION_CHECKPOINTS_DIR / "sensation_FCN-ResNet50_best.pth"
sensation_best_deeplab_path = SENSATION_CHECKPOINTS_DIR / "sensation_DeepLabV3plus_best.pth"
sensation_best_segformer_path = SENSATION_CHECKPOINTS_DIR / "sensation_SegFormer-B0_best.pth"

sensation_models["FCN-ResNet50"] = load_best_checkpoint(
    model=sensation_models["FCN-ResNet50"],
    checkpoint_path=sensation_best_fcn_path,
    device=DEVICE,
)

sensation_models["DeepLabV3+"] = load_best_checkpoint(
    model=sensation_models["DeepLabV3+"],
    checkpoint_path=sensation_best_deeplab_path,
    device=DEVICE,
)

sensation_models["SegFormer-B0"] = load_best_checkpoint(
    model=sensation_models["SegFormer-B0"],
    checkpoint_path=sensation_best_segformer_path,
    device=DEVICE,
)


In [ ]:
sensation_test_results = []
sensation_per_class_iou_rows = []
sensation_config = DATASET_CONFIGS["sensation"]

for model_name, model in sensation_models.items():
    result = evaluate_on_test_set(
        model=model,
        model_name=model_name,
        dataset_key="sensation",
        dataloader=sensation_loaders["test"],
    )

    per_class_iou = result.pop("per_class_iou")
    sensation_test_results.append(result)

    for class_id, class_iou in enumerate(per_class_iou):
        sensation_per_class_iou_rows.append({
            "dataset": sensation_config["name"],
            "model": model_name,
            "class_id": class_id,
            "class_name": sensation_config["classes"][class_id],
            "iou": class_iou,
        })

sensation_test_results_df = pd.DataFrame(sensation_test_results)
sensation_test_results_df = sensation_test_results_df.sort_values(
    by="test_mean_iou",
    ascending=False,
).reset_index(drop=True)

sensation_per_class_iou_df = pd.DataFrame(sensation_per_class_iou_rows)

display(sensation_test_results_df)
display(sensation_per_class_iou_df)

sensation_test_results_path = SENSATION_REPORTS_DIR / "sensation_test_results.csv"
sensation_per_class_path = SENSATION_REPORTS_DIR / "sensation_per_class_iou.csv"
sensation_test_results_df.to_csv(sensation_test_results_path, index=False)
sensation_per_class_iou_df.to_csv(sensation_per_class_path, index=False)

print(f"Saved test results to: {sensation_test_results_path}")
print(f"Saved per-class IoU to: {sensation_per_class_path}")


### SENSATION Qualitative and Quantitative Comparison

The qualitative grid compares the input, ground truth, and predictions from all three models. The metric chart reports the same test metrics used for CamVid.


In [ ]:
visualize_model_predictions(
    dataset_key="sensation",
    dataloader=sensation_loaders["test"],
    models=sensation_models,
    sample_count=4,
)


In [ ]:
plot_test_metric_comparison(
    results_df=sensation_test_results_df,
    dataset_key="sensation",
)


In [ ]:
display(
    sensation_test_results_df[
        [
            "model",
            "test_pixel_accuracy",
            "test_mean_iou",
            "test_dice_score",
            "test_precision",
            "test_recall",
            "inference_time_sec_per_image",
        ]
    ]
)


## Cross-Dataset Test Comparison

CamVid and SENSATION results are combined only after each dataset has been evaluated independently. This table and figure show whether the architecture ranking remains consistent across datasets with different scene distributions and annotation characteristics.


In [ ]:
combined_test_results_df = pd.concat(
    [camvid_test_results_df, sensation_test_results_df],
    ignore_index=True,
)

display(
    combined_test_results_df[
        [
            "dataset",
            "model",
            "test_mean_iou",
            "test_dice_score",
            "test_pixel_accuracy",
            "inference_time_sec_per_image",
        ]
    ].sort_values(["dataset", "test_mean_iou"], ascending=[True, False])
)


In [ ]:
def plot_cross_dataset_comparison(results_df: pd.DataFrame) -> None:
    metrics = [
        ("test_mean_iou", "Mean IoU"),
        ("test_dice_score", "Dice Score"),
    ]
    model_colors = {
        "FCN-ResNet50": "#4C78A8",
        "DeepLabV3+": "#F58518",
        "SegFormer-B0": "#54A24B",
    }

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for axis, (metric_column, title) in zip(axes, metrics):
        pivot = results_df.pivot(
            index="dataset",
            columns="model",
            values=metric_column,
        )
        colors = [model_colors[model_name] for model_name in pivot.columns]
        pivot.plot(kind="bar", ax=axis, color=colors)
        axis.set_title(title)
        axis.set_xlabel("Dataset")
        axis.set_ylabel("Score")
        axis.set_ylim(0, 1)
        axis.tick_params(axis="x", rotation=0)
        axis.grid(axis="y", alpha=0.3)

    plt.suptitle("Cross-Dataset Test Performance", fontsize=14)
    plt.tight_layout()

    output_path = REPORTS_DIR / "cross_dataset_test_comparison.png"
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()

    print(f"Saved cross-dataset comparison to: {output_path}")


plot_cross_dataset_comparison(combined_test_results_df)


## Discussion

The study compares FCN-ResNet50 and DeepLabV3+ as convolutional architectures and SegFormer-B0 as a lightweight transformer-based architecture. Each model is evaluated independently on CamVid and SENSATION using a common preprocessing, optimization, checkpoint-selection, and metric protocol within each dataset.

On CamVid, the current test results place FCN-ResNet50 first in Mean IoU at 0.5711, followed by DeepLabV3+ at 0.5664 and SegFormer-B0 at 0.5429. SegFormer-B0 nevertheless provides the lowest inference time and uses substantially fewer trainable parameters, demonstrating an accuracy-efficiency trade-off rather than a simple transformer advantage.

The SENSATION validation and test tables generated above should be used to determine whether the CamVid architecture ranking generalizes to a larger dataset. Differences should be interpreted using Mean IoU and Dice rather than pixel accuracy alone because dominant classes can conceal weak performance on smaller semantic categories.

The qualitative grids complement aggregate metrics by exposing boundary quality, thin structures, small objects, and confusion between visually related classes. Cross-dataset conclusions should consider both consistency of model ranking and changes in the size of performance differences.

SegFormer-B0 uses a MiT-B0 encoder, whereas the CNN models use ResNet50. This is therefore a comparison of complete practical architectures under a common protocol, not a backbone-controlled ablation that isolates convolution from self-attention.


## Limitations and Future Work

Both datasets contain class imbalance, and aggregate metrics may underrepresent errors on rare or spatially small classes. Per-class IoU and class-frequency analysis would provide stronger evidence about which categories benefit from each architecture.

The SENSATION class names and palette in this notebook must remain consistent with the official annotation specification. The numeric mask IDs have been preserved, but an incorrect semantic name mapping would affect interpretation even if training metrics remained numerically valid.

The experiment uses a fixed training budget and shared optimization settings for consistency. These settings may not be individually optimal for every architecture. SegFormer-B0 also has substantially fewer parameters than the ResNet50-based CNN models, so the comparison measures practical systems rather than capacity-matched architectures.

Future work could evaluate class-weighted or compound losses, stronger augmentation, per-class metrics, and capacity-matched transformer variants. Any experimental change should be applied consistently within a comparison and documented with its effect on computation and reproducibility.
